# Session 3: Retrieval Augmented Generation (RAG)
**Duration:** 2 Hours | **Day 1** | **Hands-On Workshop**

---

## Learning Objectives
By the end of this session, participants will:
1. Understand why LLMs hallucinate and how RAG solves it
2. Build a complete RAG pipeline using LangChain
3. Work with FAISS vector database for document storage and retrieval
4. Implement chunking strategies for optimal retrieval
5. Build a working **College Chatbot** that answers questions from PDF documents

---

## Prerequisites
- Ollama running with: `llama3.2`, `nomic-embed-text`
- Packages: `pip install langchain langchain-ollama langchain-community faiss-cpu pypdf`
- A sample PDF document (we will create one if needed)

---
## Part 1: Why RAG? The Hallucination Problem (Theory - 15 min)

### 1.1 The Problem with LLMs

LLMs have a fundamental limitation: **they only know what was in their training data.**

**Hallucination**: When an LLM generates confident-sounding but factually incorrect information.

Examples of hallucination:
- Citing non-existent research papers
- Making up company policies that do not exist
- Giving outdated information (training cutoff)
- Inventing statistics and dates

### 1.2 Why Do LLMs Hallucinate?
1. **No access to real-time or private data** -- trained on a fixed dataset
2. **Pattern completion** -- LLMs predict "likely" next tokens, not necessarily true ones
3. **No built-in fact verification** -- they cannot check their own answers
4. **Confidence miscalibration** -- they sound certain even when wrong

### 1.3 The RAG Solution

**Retrieval Augmented Generation (RAG)** solves this by giving the LLM access to external knowledge:

```
Traditional LLM:
  Question --> [LLM (only training data)] --> Answer (may hallucinate)

RAG Pipeline:
  Question --> [Retrieve relevant docs] --> [LLM + Retrieved context] --> Answer (grounded in facts)
```

### 1.4 RAG Architecture

```
=== INDEXING PHASE (One-time) ===

Documents (PDFs, text, web)
    |
    v
[Document Loader] --> Load raw text
    |
    v
[Text Splitter] --> Split into chunks (500-1000 chars)
    |
    v
[Embedding Model] --> Convert chunks to vectors
    |
    v
[Vector Database] --> Store vectors (FAISS/ChromaDB)


=== QUERY PHASE (Every question) ===

User Question
    |
    v
[Embedding Model] --> Convert question to vector
    |
    v
[Vector Database] --> Find top-k similar chunks
    |
    v
[Prompt Template] --> Combine question + retrieved chunks
    |
    v
[LLM] --> Generate answer grounded in retrieved context
    |
    v
Grounded Answer
```

---
## Part 2: Setting Up the RAG Components (Hands-On - 20 min)

Let's build each component step-by-step.

In [ ]:
# 2.1 First, let's create a sample document (simulating a college handbook)
# In the real workshop, participants will use actual PDF files

sample_college_handbook = """
SIDGANGA INSTITUTE OF TECHNOLOGY - STUDENT HANDBOOK 2024-25

CHAPTER 1: ABOUT THE INSTITUTION
Sidganga Institute of Technology (SIT) was established in 1963 in Tumkur, Karnataka.
It is affiliated to Visvesvaraya Technological University (VTU), Belagavi.
The institution is accredited by NAAC with 'A' grade and NBA accredited.
SIT offers undergraduate and postgraduate programs in Engineering, Technology, and Management.
The campus spans over 100 acres with modern facilities including labs, libraries, and hostels.

CHAPTER 2: DEPARTMENTS AND PROGRAMS
SIT offers the following undergraduate programs:
1. Computer Science and Engineering (CSE) - 180 seats
2. Information Science and Engineering (ISE) - 120 seats
3. Electronics and Communication Engineering (ECE) - 120 seats
4. Mechanical Engineering (ME) - 120 seats
5. Civil Engineering (CE) - 60 seats
6. Artificial Intelligence and Machine Learning (AIML) - 60 seats
7. Data Science (DS) - 60 seats

Postgraduate programs include:
1. M.Tech in Computer Science - 24 seats
2. M.Tech in Digital Communication - 18 seats
3. MBA - 120 seats
4. MCA - 60 seats

CHAPTER 3: ADMISSION PROCESS
Admission to B.E./B.Tech programs is through Karnataka CET (KCET) and COMEDK-UGET.
Management quota seats are also available with separate fee structure.
For postgraduate programs, admission is through PGCET conducted by KEA.
International students can apply through the International Admissions Office.
The academic year starts in August and ends in June.
Application deadline for KCET is usually in February.

CHAPTER 4: FEE STRUCTURE (2024-25)
B.E./B.Tech Programs:
- Government quota: Rs. 55,000 per year
- COMEDK quota: Rs. 1,20,000 per year
- Management quota: Rs. 2,50,000 per year

Hostel fees: Rs. 75,000 per year (including mess)
Bus transport: Rs. 15,000 per year

Scholarships available:
- State Government scholarships for SC/ST/OBC students
- Merit scholarships for top 10 rankers in each department
- Industry-sponsored scholarships from TCS, Infosys, Wipro

CHAPTER 5: ACADEMIC REGULATIONS
SIT follows the VTU academic regulations with the following key points:
- Minimum 75% attendance required to appear for exams
- Internal assessment: 50 marks (CIE)
- Semester end exam: 50 marks (SEE)
- Students must pass both CIE and SEE components
- Grading system: CGPA on a 10-point scale
- Minimum CGPA of 4.0 required to pass
- Maximum of 4 years additional (8 years total) to complete B.E.
- Research projects are mandatory in 7th and 8th semesters

CHAPTER 6: PLACEMENT AND CAREER SERVICES
SIT has a dedicated Training and Placement Cell.
Placement statistics for 2023-24:
- Overall placement rate: 85%
- CSE department placement rate: 95%
- Average package: Rs. 6.5 LPA
- Highest package: Rs. 42 LPA (Google)
- Top recruiters: TCS, Infosys, Wipro, Cognizant, Amazon, Google, Microsoft
- Pre-placement training starts from 5th semester
- Mock interviews and aptitude tests conducted regularly
- Internship support for 6th and 7th semester students

CHAPTER 7: FACILITIES
Library: Central library with 1,00,000+ books and digital access to IEEE, Springer, Elsevier.
Hostel: Separate hostels for boys and girls with 24/7 security and WiFi.
Sports: Cricket ground, football field, basketball court, indoor games facility.
Labs: State-of-the-art labs including AI/ML lab, IoT lab, Cloud computing lab.
Cafeteria: Multiple food courts with subsidized meals.
Medical: On-campus health center with ambulance facility.
Transport: Bus service covering Tumkur city and nearby areas.

CHAPTER 8: STUDENT CLUBS AND ACTIVITIES
- Technical clubs: Coding Club, Robotics Club, AI Club, Cybersecurity Club
- Cultural clubs: Music Club, Dance Club, Drama Club, Art Club
- Annual tech fest: "TechSpark" held in February
- Annual cultural fest: "SIT Utsav" held in March
- NSS and NCC units available
- Industry visits and guest lectures organized monthly

CHAPTER 9: CONTACT INFORMATION
Address: Sidganga Institute of Technology, SH 4, Tumkur - 572103, Karnataka
Phone: 0816-2282722
Email: info@sit.ac.in
Website: www.sit.ac.in
Admission enquiry: admissions@sit.ac.in
Placement cell: placement@sit.ac.in
"""

# Save to a text file (simulating document upload)
with open("../data/college_handbook.txt", "w", encoding="utf-8") as f:
    f.write(sample_college_handbook)

print(f"Document saved! Total characters: {len(sample_college_handbook)}")
print(f"Approximate pages: {len(sample_college_handbook) // 3000 + 1}")

In [ ]:
# 2.2 Document Loading
from langchain_community.document_loaders import TextLoader

# Load the document
loader = TextLoader("../data/college_handbook.txt", encoding="utf-8")
documents = loader.load()

print(f"Loaded {len(documents)} document(s)")
print(f"Total content length: {len(documents[0].page_content)} characters")
print(f"\nFirst 200 characters:")
print(documents[0].page_content[:200])

In [ ]:
# NOTE: For PDF files, you would use:
# from langchain_community.document_loaders import PyPDFLoader
# loader = PyPDFLoader("path/to/your/document.pdf")
# documents = loader.load()

---
## Part 3: Chunking Strategies (Theory + Hands-On - 20 min)

### 3.1 Why Chunk Documents?

- LLMs have **context window limits** (e.g., 4K-128K tokens)
- Embeddings work best on **focused, coherent text** (not entire documents)
- Smaller chunks allow **more precise retrieval**

### 3.2 Chunking Strategies

| Strategy | Description | Best For |
|----------|-------------|----------|
| **Fixed Size** | Split every N characters | Simple, fast |
| **Recursive Character** | Split by separators (\n\n, \n, space) | Most use cases (recommended) |
| **Semantic** | Split by topic/meaning boundaries | High-quality retrieval |
| **Sentence** | Split by sentences | Conversation data |

### Key Parameters:
- **chunk_size**: Maximum characters per chunk (typically 500-1500)
- **chunk_overlap**: Characters shared between adjacent chunks (typically 100-200)
  - Overlap prevents information loss at chunk boundaries

In [ ]:
# 3.3 Experimenting with Different Chunking Strategies
from langchain.text_splitter import (
    RecursiveCharacterTextSplitter,
    CharacterTextSplitter
)

# Strategy 1: Fixed Size Chunking
fixed_splitter = CharacterTextSplitter(
    separator="",
    chunk_size=500,
    chunk_overlap=50
)
fixed_chunks = fixed_splitter.split_documents(documents)
print(f"Fixed Size Chunking: {len(fixed_chunks)} chunks")

# Strategy 2: Recursive Character Splitting (RECOMMENDED)
recursive_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]  # Try these separators in order
)
recursive_chunks = recursive_splitter.split_documents(documents)
print(f"Recursive Splitting: {len(recursive_chunks)} chunks")

# Strategy 3: Larger chunks for more context
large_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    separators=["\n\n", "\n", ". ", " ", ""]
)
large_chunks = large_splitter.split_documents(documents)
print(f"Large Chunks: {len(large_chunks)} chunks")

In [ ]:
# 3.4 Inspect the chunks
print("=== Recursive Chunks (First 5) ===")
for i, chunk in enumerate(recursive_chunks[:5]):
    print(f"\n--- Chunk {i+1} (Length: {len(chunk.page_content)} chars) ---")
    print(chunk.page_content[:200] + "..." if len(chunk.page_content) > 200 else chunk.page_content)
    print()

In [ ]:
# We will use recursive chunking with 500 char size for our RAG pipeline
# This is the recommended default for most use cases

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"Final chunk count: {len(chunks)}")
print(f"Average chunk size: {sum(len(c.page_content) for c in chunks) / len(chunks):.0f} chars")

---
## Part 4: Building the Vector Database (Hands-On - 20 min)

### 4.1 What is FAISS?

**FAISS** (Facebook AI Similarity Search) is a library for efficient similarity search.
- Developed by Meta AI Research
- Supports billions of vectors
- GPU-accelerated (optional)
- Perfect for RAG applications

How it works:
```
Text chunks --> Embedding model --> Vectors --> FAISS index
                                                    |
Query --> Embedding model --> Query vector --> FAISS search --> Top-k similar chunks
```

In [ ]:
# 4.2 Create Embeddings and Build FAISS Index
from langchain_ollama import OllamaEmbeddings
from langchain_community.vectorstores import FAISS

# Initialize the embedding model
embeddings = OllamaEmbeddings(model="nomic-embed-text")

# Test the embedding model
test_embedding = embeddings.embed_query("What is the placement rate?")
print(f"Embedding dimension: {len(test_embedding)}")
print(f"First 5 values: {test_embedding[:5]}")

In [ ]:
# 4.3 Build the FAISS vector store from our document chunks
# This step converts all chunks to embeddings and stores them

print("Building FAISS index from document chunks...")
print(f"Number of chunks to embed: {len(chunks)}")

vectorstore = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

print(f"FAISS index built successfully!")
print(f"Total vectors in index: {vectorstore.index.ntotal}")

In [ ]:
# 4.4 Test retrieval -- find relevant chunks for a query

def test_retrieval(query, k=3):
    """Test retrieval from the vector store."""
    results = vectorstore.similarity_search_with_score(query, k=k)

    print(f"Query: \"{query}\"")
    print(f"Found {len(results)} relevant chunks:\n")

    for i, (doc, score) in enumerate(results, 1):
        print(f"--- Result {i} (Distance: {score:.4f}) ---")
        print(doc.page_content[:200])
        print()

# Test with different queries
test_retrieval("What is the placement rate at SIT?")
print("=" * 60)
test_retrieval("How much does hostel cost?")
print("=" * 60)
test_retrieval("What clubs are available for students?")

In [ ]:
# 4.5 Save and Load FAISS index (persistence)

# Save the index
vectorstore.save_local("../data/faiss_college_index")
print("FAISS index saved to disk!")

# To load later:
# loaded_vectorstore = FAISS.load_local(
#     "../data/faiss_college_index",
#     embeddings,
#     allow_dangerous_deserialization=True
# )
# print("FAISS index loaded from disk!")

---
## Part 5: Building the Complete RAG Pipeline (Hands-On - 30 min)

Now we connect all the pieces:
1. User asks a question
2. Retrieve relevant chunks from FAISS
3. Format a prompt with the context
4. LLM generates an answer grounded in the context

In [ ]:
# 5.1 Create the RAG chain using LangChain
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

# Initialize the LLM
llm = ChatOllama(model="llama3.2", temperature=0)

# Create a retriever from our vector store
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4}  # Retrieve top 4 chunks
)

# RAG Prompt Template
rag_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful college information assistant for Sidganga Institute of Technology (SIT).

RULES:
1. Answer ONLY based on the provided context
2. If the context does not contain the answer, say "I don't have information about that in my documents."
3. Be specific and cite numbers/details from the context when available
4. Keep answers concise but complete
5. Do NOT make up information"""),
    ("human", """Context from college documents:
{context}

Question: {question}

Answer based on the context above:""")
])

print("RAG components initialized!")

In [ ]:
# 5.2 Build the RAG Chain

def format_docs(docs):
    """Format retrieved documents into a single context string."""
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

# Create the RAG chain using LangChain Expression Language (LCEL)
rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | rag_prompt
    | llm
    | StrOutputParser()
)

print("RAG chain built successfully!")
print("\nPipeline: Question --> Retriever --> Format --> Prompt --> LLM --> Answer")

In [ ]:
# 5.3 Test the RAG Chatbot!

def ask_college_bot(question):
    """Ask a question to the college chatbot."""
    print(f"Q: {question}")
    print("-" * 50)
    answer = rag_chain.invoke(question)
    print(f"A: {answer}")
    print("=" * 60)
    print()

# Test with various questions
ask_college_bot("What is the placement rate for CSE department?")

In [ ]:
ask_college_bot("What is the fee structure for B.Tech programs?")

In [ ]:
ask_college_bot("What are the hostel facilities?")

In [ ]:
ask_college_bot("What is the minimum attendance required?")

In [ ]:
ask_college_bot("What scholarships are available?")

In [ ]:
# Test hallucination prevention -- ask about something NOT in the documents
ask_college_bot("What is the PhD program fee structure?")

In [ ]:
# 5.4 Enhanced RAG -- Show sources with the answer

def ask_with_sources(question):
    """Ask a question and show which document chunks were used."""
    # Retrieve documents
    retrieved_docs = retriever.invoke(question)
    context = format_docs(retrieved_docs)

    # Generate answer
    messages = rag_prompt.format_messages(context=context, question=question)
    answer = llm.invoke(messages)

    print(f"Q: {question}")
    print("=" * 50)
    print(f"A: {answer.content}")
    print("\n--- Sources Used ---")
    for i, doc in enumerate(retrieved_docs, 1):
        print(f"\nSource {i} (first 150 chars):")
        print(f"  {doc.page_content[:150]}...")
    print()

ask_with_sources("What are the top recruiting companies at SIT?")

---
## Part 6: Interactive Chatbot Loop (Hands-On - 15 min)

Let's create a conversational interface for our chatbot.

In [ ]:
# 6.1 Interactive Chatbot with conversation history

from langchain_core.prompts import MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

# Create a conversational RAG prompt
conversational_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a friendly and helpful college information assistant for SIT, Tumkur.
    Answer questions based on the provided context.
    If the context does not contain the answer, say so honestly.
    Refer to previous conversation context when relevant."""),
    MessagesPlaceholder(variable_name="chat_history"),
    ("human", """Context from college documents:
{context}

Question: {question}""")
])

# Store conversation history
chat_history = []

def chat(question):
    """Chat with the college bot, maintaining conversation history."""
    # Retrieve relevant context
    docs = retriever.invoke(question)
    context = format_docs(docs)

    # Generate response
    messages = conversational_prompt.format_messages(
        context=context,
        question=question,
        chat_history=chat_history
    )
    response = llm.invoke(messages)

    # Update history
    chat_history.append(HumanMessage(content=question))
    chat_history.append(AIMessage(content=response.content))

    print(f"You: {question}")
    print(f"Bot: {response.content}")
    print("-" * 40)

# Simulate a conversation
chat("How many seats are there in CSE?")
chat("What about the placement rate for that department?")
chat("And what is the highest package offered?")

---
## Part 7: RAG with PDF Files (Bonus - if time permits)

Here is how to use the same pipeline with actual PDF documents:

In [ ]:
# 7.1 RAG with PDF -- Template code (uncomment and use with your own PDFs)

# from langchain_community.document_loaders import PyPDFLoader
#
# # Load PDF
# pdf_loader = PyPDFLoader("path/to/your/document.pdf")
# pdf_documents = pdf_loader.load()
#
# print(f"Loaded {len(pdf_documents)} pages from PDF")
#
# # Split into chunks
# pdf_chunks = text_splitter.split_documents(pdf_documents)
# print(f"Created {len(pdf_chunks)} chunks")
#
# # Build vector store
# pdf_vectorstore = FAISS.from_documents(pdf_chunks, embeddings)
#
# # Create retriever and chain (same as before)
# pdf_retriever = pdf_vectorstore.as_retriever(search_kwargs={"k": 4})
# pdf_chain = (
#     {"context": pdf_retriever | format_docs, "question": RunnablePassthrough()}
#     | rag_prompt | llm | StrOutputParser()
# )
#
# # Ask questions about your PDF!
# print(pdf_chain.invoke("Your question here"))

print("PDF RAG template ready -- uncomment and use with your own PDFs")

---
## Exercises

### Exercise 1: Chunk Size Experiment
Build the same RAG pipeline with chunk sizes of 200, 500, and 1000. Ask the same 5 questions and compare answer quality. Which chunk size gives the best results?

### Exercise 2: Multi-Document RAG
Add a second document (e.g., department-specific syllabus) to the vector store. How does retrieval work across multiple documents?

### Exercise 3: Hallucination Test
Create a list of 10 questions -- 5 answerable from the documents, 5 not answerable. Evaluate how well the RAG system handles both categories.

---

## Summary

### Key Takeaways:
1. **RAG solves hallucination** by grounding LLM responses in retrieved documents
2. **Chunking strategy** significantly affects retrieval quality -- Recursive Character Splitting is the best default
3. **FAISS** provides fast, local vector similarity search
4. **LangChain LCEL** makes it easy to compose RAG pipelines
5. The prompt template is critical -- it tells the LLM to ONLY use the provided context
6. **Source attribution** builds trust by showing which documents informed the answer

### Next Session: Introduction to MCP (Model Context Protocol)
We will explore how MCP provides a standardized way for LLMs to interact with external tools and data sources.